# 05 - Build the YOLO Dataset (merge label runs, sequence-disjoint splits)

Stage D. Merges the QA-approved label runs, applies the class IDs from
`configs/experiments/dataset_v1.yaml`, splits **by sequence** (frames from one
approach never straddle splits), reserves a fraction of **clicked** sequences
as the gold test set, and writes a self-contained YOLO dataset under
`datasets/<run_id>/` (images/, labels/, dataset.yaml, provenance.json, run.json).

Run after notebook 04's QA verdict is clean. CPU-only; the slow part is copying
images (Drive -> Drive).

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json

import pandas as pd
import yaml

from vision_uss_research.datasets.yolo_export import (apply_class_groups,
                                                      force_test_for_objects,
                                                      merge_label_sources,
                                                      split_by_sequence,
                                                      write_yolo_dataset)
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import frames_window_id

CONFIG_NAME = "dataset_v1"          # or "dataset_v2" for named + open-set bucket classes
cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments" / f"{CONFIG_NAME}.yaml").read_text())
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))

# label runs to merge: auto-discover (newest batch run + all click runs),
# or set an explicit list of {"name", "boxes_csv", "frames_root", "gold"} dicts.
LABEL_SOURCES = "auto"
if LABEL_SOURCES == "auto":
    LABEL_SOURCES = []
    batch_base = ARTIFACTS / "labels" / "runs"
    for run_dir in sorted(batch_base.glob("*")) if batch_base.exists() else []:
        if (run_dir / "propagated_boxes.csv").exists():
            LABEL_SOURCES.append({"name": run_dir.name,
                                  "boxes_csv": run_dir / "propagated_boxes.csv",
                                  "frames_root": WINDOW_ROOT, "gold": False})
    click_base = ARTIFACTS / "labels" / "clicks"
    for run_dir in sorted(click_base.glob("*")) if click_base.exists() else []:
        if (run_dir / "propagated_boxes.csv").exists():
            LABEL_SOURCES.append({"name": run_dir.name,
                                  "boxes_csv": run_dir / "propagated_boxes.csv",
                                  "frames_root": WINDOW_ROOT, "gold": True})
assert LABEL_SOURCES, "no label runs found - run notebooks 02/03 first"
for s in LABEL_SOURCES:
    print(f"source: {s['name']} (gold={s['gold']})")

RUN_ID = new_run_id(cfg["stage"], cfg["tag"])
DATASET_DIR = ARTIFACTS / "datasets" / RUN_ID
start_run(DATASET_DIR, config=cfg,
          inputs=[s["name"] for s in LABEL_SOURCES], repo_root=REPO_DIR)
print(f"dataset dir: {DATASET_DIR}")

In [ ]:
merged = merge_label_sources(LABEL_SOURCES)
print(f"merged: {len(merged)} boxes, {merged['sequence_id'].nunique()} sequences")

# map bucket members (v2: cone -> high_other, ...) onto training classes
merged = apply_class_groups(merged, cfg.get("class_groups"))

# classes present but missing from the config's class map fail loudly
unmapped = sorted(set(merged["class_name"]) - set(cfg["class_ids"]))
assert not unmapped, f"add class ids for: {unmapped} (configs/experiments/dataset_v1.yaml)"

split = split_by_sequence(merged, seed=cfg["seed"],
                          gold_test_frac=cfg["split"]["gold_test_frac"],
                          auto_test_frac=cfg["split"]["auto_test_frac"],
                          val_frac=cfg["split"]["val_frac"])

# open-set experiment: holdout objects never reach train/val
holdout = cfg.get("holdout_objects") or []
split = force_test_for_objects(split, holdout)
if holdout:
    print(f"open-set holdout (test-only object types): {holdout}")

overview = (split.groupby(["target_object", "split"])
            .agg(sequences=("sequence_id", "nunique"), boxes=("x0", "size"))
            .unstack("split", fill_value=0))
display(overview)

# hard guarantees: no sequence in more than one split, and every class trains
assert (split.groupby("sequence_id")["split"].nunique() == 1).all()
train_seqs = (split[split["split"] == "train"]
              .groupby("class_name")["sequence_id"].nunique())
starved = [obj for obj in split["class_name"].unique()
           if train_seqs.get(obj, 0) == 0]
assert not starved, (
    f"classes with NO training sequences: {starved} - this dataset is built from "
    "validation-scale label runs; finish the full runs in notebooks 02/03 first")
low_volume = train_seqs[train_seqs < 20]
if len(low_volume):
    print("WARNING - classes with < 20 training sequences (thin for YOLO):")
    display(low_volume.to_frame("train_sequences"))

In [ ]:
frames_roots = {s["name"]: Path(s["frames_root"]) for s in LABEL_SOURCES}
provenance = write_yolo_dataset(split, DATASET_DIR, cfg["class_ids"],
                                frames_roots, seed=cfg["seed"])

print(f"gold test sequences : {len(provenance['gold_test_sequences'])}")
print(f"missing frames      : {len(provenance['missing_frames'])}")
finish_run(DATASET_DIR, summary={
    "boxes": int(len(split)),
    "sequences_per_split": provenance["sequences_per_split"],
    "missing_frames": len(provenance["missing_frames"]),
})
print(f"dataset ready: datasets/{RUN_ID}")
print("next: notebook 06 (training)")